# Gated Bayesian Stratigraphic-Regime Model

This notebook tests a conservative Bayesian-regime idea: fit a robust signed-delta mean, learn two residual experts, and use known-zone state/alignment descriptors to softly gate the residual correction. The gate is fit with Student-t MAP and explicit shrinkage priors so weak state evidence cannot dominate the prediction path.

## 1. Setup

In [1]:

"Configure imports, paths, and deterministic settings."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "src").exists())

import sys
sys.path.append(str(ROOT))
from src.utils.run_bundle import RunBundle

RUN_DATE = "2026-06-30"
RUN_TAG = "gated_bayesian_regime"
RUN_ID = f"{RUN_DATE}__{RUN_TAG}"
SEED = 20260630
MAX_TARGET_ROWS_PER_WELL = 120
MAX_TRAIN_ROWS_PER_FOLD = 90000
RESIDUAL_SAMPLE_PER_FOLD = 2200

NU = 3.0
HSGP_M = 8
HSGP_ELL = 3.0
HSGP_PRIOR_SCALE = 0.55
STATE_RIDGE_ALPHA = 120.0
MAX_OPT_ITER = 650
GATE_MAX_ITER = 450

GAM_N_KNOTS = 7
GAM_DEGREE = 3
GAM_ALPHA = 60.0
HUBER_ALPHA = 0.0001
HUBER_EPSILON = 1.35

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for directory in [TABLE_DIR, REPORT_DIR, RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_id": RUN_ID,
    "seed": SEED,
    "fold_column": "strict_hybrid_fold",
    "stress_fold_column": "analog_distance_fold",
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "max_train_rows_per_fold": MAX_TRAIN_ROWS_PER_FOLD,
    "current_public_best": 14.472,
    "current_local_xgb_safe": 13.4178,
    "huber_hilbert_public": 14.514,
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as handle:
    json.dump(CONFIG, handle, indent=2)

rng = np.random.default_rng(SEED)
print({"root": str(ROOT), "run_id": RUN_ID})


{'root': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling', 'run_id': '2026-06-30__gated_bayesian_regime'}


## 2. Data And Feature Construction

In [2]:

"Build sampled target-zone rows with slope-core features, known-zone state summaries, and fold assignments."

folds = pd.read_csv(TABLE_DIR / "validation__strict_cv_fold_assignments__2026-06-26.csv")
descriptor_path = TABLE_DIR / "model__typewell_alignment_feature_descriptors__2026-06-23.csv"
descriptors = pd.read_csv(descriptor_path) if descriptor_path.exists() else pd.DataFrame({"wellname": folds["wellname"]})
well_meta = folds.merge(descriptors, on="wellname", how="left").drop_duplicates("wellname").reset_index(drop=True)

descriptor_cols = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "descriptor_cluster",
]
smooth_features = [
    "distance_from_boundary_md", "eval_progress", "MD", "Z", "md_norm", "dX_from_boundary", "dY_from_boundary", "dZ_from_boundary",
    "horizontal_dist_from_boundary", "tvt_slope_last_10", "tvt_slope_last_25", "tvt_slope_last_50", "tvt_slope_last_100",
    "z_slope_last_25", "z_slope_last_50", "z_slope_last_100", "tvt_slope_last_10_projected_delta",
    "tvt_slope_last_25_projected_delta", "tvt_slope_last_50_projected_delta", "tvt_slope_last_100_projected_delta",
]
linear_features = ["last_known_tvt"]
state_feature_cols = [
    "state_known_tvt_last", "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last", "state_known_gr_iqr",
    "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]
typewell_gate_cols = [
    "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr", "typewell_best_lag_mae",
    "selected_analog_distance", "extrapolation_stress_score", "analog_stress_score", "combined_stress_score",
]
gate_feature_cols = list(dict.fromkeys(state_feature_cols + typewell_gate_cols + ["distance_log1p", "eval_progress", "abs_projected_delta_10"]))
feature_columns = list(dict.fromkeys(smooth_features + linear_features))
hsgp_coordinates = ["distance_log1p", "eval_progress", "abs_projected_delta_10"]


def load_horizontal_well(path):
    df = pd.read_csv(path)
    df.index.name = "row_idx"
    df["row_idx"] = df.index
    return df


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def deterministic_row_cap(frame, max_rows):
    if len(frame) <= max_rows:
        return frame
    positions = np.linspace(0, len(frame) - 1, max_rows).astype(int)
    return frame.iloc[positions].copy()


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0 = float(md.iloc[start_pos])
    x1 = float(md.iloc[end_pos])
    y0 = float(values.iloc[start_pos])
    y1 = float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def build_features_for_well(meta_row):
    well = meta_row["wellname"]
    path = TRAIN_DIR / f"{well}__horizontal_well.csv"
    if not path.exists():
        return pd.DataFrame()
    horizontal = load_horizontal_well(path).sort_values("MD").reset_index(drop=True)
    known_mask = horizontal["TVT_input"].notna()
    target_mask = horizontal["TVT_input"].isna() & horizontal["TVT"].notna()
    if int(target_mask.sum()) == 0 or int(known_mask.sum()) == 0:
        return pd.DataFrame()

    tvt_input = safe_numeric(horizontal["TVT_input"])
    md = safe_numeric(horizontal["MD"])
    x = safe_numeric(horizontal["X"])
    y = safe_numeric(horizontal["Y"])
    z = safe_numeric(horizontal["Z"])
    gr = safe_numeric(horizontal["GR"]).clip(0, 300)

    known_positions = np.flatnonzero(known_mask.to_numpy())
    last_known_pos = int(known_positions[-1])
    known_gr = gr.iloc[:last_known_pos + 1]
    known_gr_finite = known_gr.dropna()

    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    last_known_md = float(md.iloc[last_known_pos])
    last_known_x = float(x.iloc[last_known_pos])
    last_known_y = float(y.iloc[last_known_pos])
    last_known_z = float(z.iloc[last_known_pos])

    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{window}": window_slope(md, tvt_input, last_known_pos, window) for window in slope_windows}
    z_slopes = {f"z_slope_last_{window}": window_slope(md, z, last_known_pos, window) for window in slope_windows}
    horizontal_step = np.sqrt(x.diff().fillna(0.0) ** 2 + y.diff().fillna(0.0) ** 2)
    horizontal_dist = horizontal_step.cumsum()

    known_md_span_state = float(md.iloc[last_known_pos] - md.iloc[0]) if last_known_pos > 0 else 0.0
    known_tvt_delta_state = float(tvt_input.iloc[last_known_pos] - tvt_input.iloc[0]) if last_known_pos > 0 else 0.0
    known_tvt_slope_state = known_tvt_delta_state / max(abs(known_md_span_state), 1.0)
    known_xy_span_state = float(np.hypot(x.iloc[last_known_pos] - x.iloc[0], y.iloc[last_known_pos] - y.iloc[0])) if last_known_pos > 0 else 0.0
    known_z_drop_state = float(z.iloc[last_known_pos] - z.iloc[0]) if last_known_pos > 0 else 0.0
    known_gr_coverage_state = float(known_gr.notna().mean()) if len(known_gr) else 0.0
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    known_gr_last_state = float(known_gr_finite.iloc[-1]) if len(known_gr_finite) else gr_fill
    known_gr_iqr_state = float(known_gr_finite.quantile(0.75) - known_gr_finite.quantile(0.25)) if len(known_gr_finite) >= 4 else 0.0

    target_positions = deterministic_sample_positions(target_mask, MAX_TARGET_ROWS_PER_WELL)
    all_target_positions = np.flatnonzero(target_mask.to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(all_target_positions)}
    eval_denominator = max(len(all_target_positions) - 1, 1)
    rows = []
    for pos in target_positions:
        distance_md = float(md.iloc[pos] - last_known_md)
        tvt_slope_10 = tvt_slopes["tvt_slope_last_10"]
        projected_delta_10 = tvt_slope_10 * distance_md if np.isfinite(tvt_slope_10) else np.nan
        row = {
            "wellname": well,
            "row_idx": int(horizontal.index[pos]),
            "TVT": float(horizontal["TVT"].iloc[pos]),
            "last_known_tvt": last_known_tvt,
            "MD": float(md.iloc[pos]),
            "Z": float(z.iloc[pos]),
            "GR": float(gr.iloc[pos]) if np.isfinite(gr.iloc[pos]) else gr_fill,
            "delta_tvt_from_boundary": float(horizontal["TVT"].iloc[pos]) - last_known_tvt,
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "distance_log1p": float(np.log1p(max(distance_md, 0.0))),
            "eval_progress": float(eval_rank[int(pos)] / eval_denominator),
            "dX_from_boundary": float(x.iloc[pos] - last_known_x),
            "dY_from_boundary": float(y.iloc[pos] - last_known_y),
            "dZ_from_boundary": float(z.iloc[pos] - last_known_z),
            "horizontal_dist_from_boundary": float(horizontal_dist.iloc[pos] - horizontal_dist.iloc[last_known_pos]),
            "abs_projected_delta_10": float(abs(projected_delta_10)) if np.isfinite(projected_delta_10) else np.nan,
            "state_known_tvt_last": last_known_tvt,
            "state_known_tvt_slope": known_tvt_slope_state,
            "state_known_gr_coverage": known_gr_coverage_state,
            "state_known_gr_last": known_gr_last_state,
            "state_known_gr_iqr": known_gr_iqr_state,
            "state_known_xy_span": known_xy_span_state,
            "state_known_z_drop": known_z_drop_state,
            "state_known_md_span": known_md_span_state,
        }
        for name, value in tvt_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
            row[f"{name}_projected_delta"] = row[name] * distance_md if np.isfinite(row[name]) else np.nan
        for name, value in z_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
        for col in [CONFIG["fold_column"], CONFIG["stress_fold_column"], "descriptor_cluster_fold", "extrapolation_fold"]:
            row[col] = int(meta_row[col]) if col in meta_row and pd.notna(meta_row[col]) else -1
        for col in descriptor_cols:
            value = meta_row.get(col, np.nan)
            row[col] = float(value) if pd.notna(value) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


frames = [build_features_for_well(row) for row in well_meta.to_dict(orient="records")]
feature_frame = pd.concat([frame for frame in frames if not frame.empty], ignore_index=True)
feature_frame[gate_feature_cols] = feature_frame[gate_feature_cols].replace([np.inf, -np.inf], np.nan)
feature_preview_path = TABLE_DIR / f"model__gated_bayesian_regime_feature_preview__{RUN_DATE}.csv"
feature_frame.head(500).to_csv(feature_preview_path, index=False)
shutil.copy2(feature_preview_path, RUN_DIR / feature_preview_path.name)

print({"rows": int(len(feature_frame)), "wells": int(feature_frame["wellname"].nunique()), "folds": sorted(feature_frame[CONFIG["fold_column"]].unique())})
feature_frame[["wellname", "row_idx", "TVT", "delta_tvt_from_boundary", "distance_log1p", "eval_progress"] + state_feature_cols].head()


{'rows': 92760, 'wells': 773, 'folds': [np.int64(1), np.int64(2), np.int64(3)]}


,wellname,row_idx,TVT,delta_tvt_from_boundary,distance_log1p,eval_progress,state_known_tvt_last,state_known_tvt_slope,state_known_gr_coverage,state_known_gr_last,state_known_gr_iqr,state_known_xy_span,state_known_z_drop,state_known_md_span
0,000d7d20,1442,11747.38,0.01,0.693147,0.000000,11747.37,0.354858,0.693481,106.668203,28.970525,1189.689817,-476.51,1441.0
1,000d7d20,1474,11747.51,0.14,3.526361,0.008344,11747.37,0.354858,0.693481,106.668203,28.970525,1189.689817,-476.51,1441.0
2,000d7d20,1506,11747.61,0.24,4.189655,0.016688,11747.37,0.354858,0.693481,106.668203,28.970525,1189.689817,-476.51,1441.0
3,000d7d20,1538,11748.36,0.99,4.584967,0.025033,11747.37,0.354858,0.693481,106.668203,28.970525,1189.689817,-476.51,1441.0
4,000d7d20,1570,11749.02,1.65,4.867534,0.033377,11747.37,0.354858,0.693481,106.668203,28.970525,1189.689817,-476.51,1441.0


## 3. Mean, Expert, And Gate Models

In [3]:

"Define robust mean, residual experts, and Student-t MAP soft-gate helpers."

def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def metric_row(y_true, y_pred):
    return {"rmse": rmse(y_true, y_pred), "mae": float(mean_absolute_error(y_true, y_pred)), "r2": float(r2_score(y_true, y_pred))}


def make_huber_model():
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", HuberRegressor(alpha=HUBER_ALPHA, epsilon=HUBER_EPSILON, max_iter=1200))])


def make_gam_model():
    transformer = ColumnTransformer(
        transformers=[
            ("smooth", Pipeline([("imputer", SimpleImputer(strategy="median")), ("spline", SplineTransformer(n_knots=GAM_N_KNOTS, degree=GAM_DEGREE, include_bias=False)), ("scale", StandardScaler())]), smooth_features),
            ("linear", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), linear_features),
        ],
        remainder="drop",
    )
    return Pipeline([("features", transformer), ("model", Ridge(alpha=GAM_ALPHA))])


def fit_coord_preprocessor(frame, columns):
    medians = frame[columns].median(numeric_only=True)
    filled = frame[columns].fillna(medians)
    means = filled.mean()
    stds = filled.std().replace(0.0, 1.0).fillna(1.0)
    return medians, means, stds


def apply_coord_preprocessor(frame, columns, medians, means, stds):
    filled = frame[columns].fillna(medians)
    return ((filled - means) / stds).to_numpy(float)


def hilbert_basis_1d(values, m=HSGP_M, ell=HSGP_ELL):
    clipped = np.clip(values, -ell, ell)
    shifted = (clipped + ell) / (2.0 * ell)
    return np.column_stack([np.sqrt(1.0 / ell) * np.sin(np.pi * j * shifted) for j in range(1, m + 1)])


def additive_hilbert_design(frame, model, coordinates):
    coords = apply_coord_preprocessor(frame, coordinates, model["medians"], model["means"], model["stds"])
    blocks = [hilbert_basis_1d(coords[:, idx]) for idx in range(coords.shape[1])]
    return np.column_stack(blocks)


def softplus(value):
    return np.logaddexp(0.0, value)


def sigmoid(value):
    return 1.0 / (1.0 + np.exp(-np.clip(value, -35, 35)))


def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def residual_student_t_objective(params, X, y):
    intercept = params[0]
    beta = params[1:-1]
    sigma = softplus(params[-1]) + 1e-6
    residual = y - (intercept + X @ beta)
    scaled = residual / sigma
    nll = np.sum(np.log(sigma) + 0.5 * (NU + 1.0) * np.log1p((scaled ** 2) / NU))
    return float(nll + 0.5 * np.sum((beta / HSGP_PRIOR_SCALE) ** 2) + 0.5 * (intercept / 1.0) ** 2 + 0.5 * (sigma / 1.5) ** 2)


def fit_hsgp_residual_model(train_frame, residual, coordinates):
    medians, means, stds = fit_coord_preprocessor(train_frame, coordinates)
    model = {"coordinates": coordinates, "medians": medians, "means": means, "stds": stds}
    X = additive_hilbert_design(train_frame, model, coordinates)
    y_mean = float(np.mean(residual))
    y_std = float(np.std(residual) + 1e-9)
    y = (residual - y_mean) / y_std
    start = np.zeros(X.shape[1] + 2, dtype=float)
    start[-1] = np.log(np.expm1(1.0))
    result = minimize(residual_student_t_objective, start, args=(X, y), method="L-BFGS-B", options={"maxiter": MAX_OPT_ITER, "ftol": 1e-8, "maxls": 50})
    model.update({"params": result.x, "y_mean": y_mean, "y_std": y_std, "loss": float(result.fun), "success": bool(result.success), "nit": int(result.nit), "n_basis": int(X.shape[1])})
    return model


def predict_hsgp_residual(model, frame):
    X = additive_hilbert_design(frame, model, model["coordinates"])
    params = model["params"]
    pred_scaled = params[0] + X @ params[1:-1]
    return pred_scaled * model["y_std"] + model["y_mean"]


def make_scaled_matrix(train_frame, valid_frame, columns):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(imputer.fit_transform(train_frame[columns]))
    X_valid = scaler.transform(imputer.transform(valid_frame[columns]))
    return X_train, X_valid


def fit_state_residual_model(train_frame, residual):
    model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("ridge", Ridge(alpha=STATE_RIDGE_ALPHA))])
    model.fit(train_frame[gate_feature_cols], residual)
    return model


def gate_objective(params, X, component_preds, residual, prior_scale, shrink_prior):
    n_features = X.shape[1]
    n_components = component_preds.shape[1]
    size_w = n_features * n_components
    W = params[:size_w].reshape(n_features, n_components)
    b = params[size_w:size_w + n_components]
    shrink_raw = params[size_w + n_components]
    sigma = softplus(params[-1]) + 1e-6
    weights = softmax(X @ W + b)
    shrink = sigmoid(shrink_raw)
    pred = shrink * np.sum(weights * component_preds, axis=1)
    error = residual - pred
    scaled = error / sigma
    nll = np.sum(np.log(sigma) + 0.5 * (NU + 1.0) * np.log1p((scaled ** 2) / NU))
    shrink_prior_logit = np.log(shrink_prior / (1.0 - shrink_prior))
    return float(nll + 0.5 * np.sum((W / prior_scale) ** 2) + 0.5 * np.sum((b / 1.0) ** 2) + 0.5 * ((shrink_raw - shrink_prior_logit) / 0.75) ** 2 + 0.5 * (sigma / 1.5) ** 2)


def fit_gate_model(X_train, component_train, residual, prior_scale, shrink_prior):
    n_features = X_train.shape[1]
    n_components = component_train.shape[1]
    start = np.zeros(n_features * n_components + n_components + 2, dtype=float)
    start[-2] = np.log(shrink_prior / (1.0 - shrink_prior))
    start[-1] = np.log(np.expm1(np.std(residual) + 1e-6))
    result = minimize(gate_objective, start, args=(X_train, component_train, residual, prior_scale, shrink_prior), method="L-BFGS-B", options={"maxiter": GATE_MAX_ITER, "ftol": 1e-8, "maxls": 50})
    return {"params": result.x, "prior_scale": prior_scale, "shrink_prior": shrink_prior, "success": bool(result.success), "loss": float(result.fun), "nit": int(result.nit), "n_features": n_features, "n_components": n_components}


def predict_gate_model(model, X, component_preds):
    params = model["params"]
    n_features = model["n_features"]
    n_components = model["n_components"]
    size_w = n_features * n_components
    W = params[:size_w].reshape(n_features, n_components)
    b = params[size_w:size_w + n_components]
    shrink = sigmoid(params[size_w + n_components])
    weights = softmax(X @ W + b)
    pred = shrink * np.sum(weights * component_preds, axis=1)
    return pred, weights, shrink


candidate_specs = [
    {"candidate": "huber_mean_only", "mode": "mean", "mean": "huber"},
    {"candidate": "gam_mean_only", "mode": "mean", "mean": "gam"},
    {"candidate": "huber_plus_hsgp_s025", "mode": "fixed", "mean": "huber", "component": "hsgp", "shrink": 0.25},
    {"candidate": "huber_plus_state_s025", "mode": "fixed", "mean": "huber", "component": "state", "shrink": 0.25},
    {"candidate": "huber_plus_gate_conservative", "mode": "gate", "mean": "huber", "prior_scale": 0.25, "shrink_prior": 0.25},
    {"candidate": "huber_plus_gate_medium", "mode": "gate", "mean": "huber", "prior_scale": 0.50, "shrink_prior": 0.45},
    {"candidate": "huber_plus_gate_open", "mode": "gate", "mean": "huber", "prior_scale": 0.80, "shrink_prior": 0.60},
]
pd.DataFrame(candidate_specs)


,candidate,mode,mean,component,shrink,prior_scale,shrink_prior
0,huber_mean_only,mean,huber,NaN,NaN,NaN,NaN
1,gam_mean_only,mean,gam,NaN,NaN,NaN,NaN
2,huber_plus_hsgp_s025,fixed,huber,hsgp,0.25,NaN,NaN
3,huber_plus_state_s025,fixed,huber,state,0.25,NaN,NaN
4,huber_plus_gate_conservative,gate,huber,NaN,NaN,0.25,0.25
5,huber_plus_gate_medium,gate,huber,NaN,NaN,0.50,0.45
6,huber_plus_gate_open,gate,huber,NaN,NaN,0.80,0.60


## 4. Strict-CV Regime Sweep

In [4]:

"Fit mean models, residual experts, and Bayesian gates under strict-hybrid folds."

start_time = time.time()
fold_rows = []
residual_parts = []
gate_diag_rows = []
weight_parts = []
component_names = ["zero", "hsgp", "state", "blend"]

for fold in sorted(feature_frame[CONFIG["fold_column"]].unique()):
    valid_mask = feature_frame[CONFIG["fold_column"]].eq(fold)
    train_frame_full = feature_frame.loc[~valid_mask].copy()
    valid_frame = feature_frame.loc[valid_mask].copy()
    train_frame = deterministic_row_cap(train_frame_full, MAX_TRAIN_ROWS_PER_FOLD)

    y_train_delta = train_frame["delta_tvt_from_boundary"].to_numpy(float)
    y_valid_tvt = valid_frame["TVT"].to_numpy(float)
    valid_last_known = valid_frame["last_known_tvt"].to_numpy(float)

    mean_models = {"huber": make_huber_model(), "gam": make_gam_model()}
    for name, model in mean_models.items():
        model.fit(train_frame[feature_columns], y_train_delta)

    huber_train_delta = mean_models["huber"].predict(train_frame[feature_columns])
    train_residual = y_train_delta - huber_train_delta
    hsgp_model = fit_hsgp_residual_model(train_frame, train_residual, hsgp_coordinates)
    hsgp_train = predict_hsgp_residual(hsgp_model, train_frame)
    hsgp_valid = predict_hsgp_residual(hsgp_model, valid_frame)
    state_model = fit_state_residual_model(train_frame, train_residual)
    state_train = state_model.predict(train_frame[gate_feature_cols])
    state_valid = state_model.predict(valid_frame[gate_feature_cols])
    X_gate_train, X_gate_valid = make_scaled_matrix(train_frame, valid_frame, gate_feature_cols)
    component_train = np.column_stack([np.zeros(len(train_frame)), hsgp_train, state_train, 0.5 * hsgp_train + 0.5 * state_train])
    component_valid = np.column_stack([np.zeros(len(valid_frame)), hsgp_valid, state_valid, 0.5 * hsgp_valid + 0.5 * state_valid])

    for spec in candidate_specs:
        mean_delta = mean_models[spec["mean"]].predict(valid_frame[feature_columns])
        correction = np.zeros(len(valid_frame), dtype=float)
        gate_shrink = 0.0
        gate_success = True
        gate_loss = np.nan
        gate_iterations = 0
        mean_weights = {f"weight_{name}": np.nan for name in component_names}
        weights = np.full((len(valid_frame), len(component_names)), np.nan)

        if spec["mode"] == "fixed":
            source = hsgp_valid if spec["component"] == "hsgp" else state_valid
            correction = spec["shrink"] * source
            gate_shrink = float(spec["shrink"])
        elif spec["mode"] == "gate":
            gate_model = fit_gate_model(X_gate_train, component_train, train_residual, spec["prior_scale"], spec["shrink_prior"])
            correction, weights, gate_shrink = predict_gate_model(gate_model, X_gate_valid, component_valid)
            gate_success = gate_model["success"]
            gate_loss = gate_model["loss"]
            gate_iterations = gate_model["nit"]
            for idx, name in enumerate(component_names):
                mean_weights[f"weight_{name}"] = float(np.mean(weights[:, idx]))
            gate_diag_rows.append({"fold": int(fold), "candidate": spec["candidate"], "prior_scale": spec["prior_scale"], "shrink_prior": spec["shrink_prior"], "learned_shrink": float(gate_shrink), "optimizer_success": gate_success, "optimizer_iterations": gate_iterations, "loss": gate_loss, **mean_weights})

        pred_delta = mean_delta + correction
        pred_tvt = valid_last_known + pred_delta
        baseline_tvt = valid_last_known
        metrics = metric_row(y_valid_tvt, pred_tvt)
        hold_rmse = rmse(y_valid_tvt, baseline_tvt)
        fold_rows.append({"fold": int(fold), "fold_column": CONFIG["fold_column"], "candidate": spec["candidate"], "mode": spec["mode"], "mean_model": spec["mean"], "n_train_rows": int(len(train_frame)), "n_valid_rows": int(len(valid_frame)), "correction_mean": float(np.mean(correction)), "correction_sd": float(np.std(correction)), "gate_shrink": float(gate_shrink), "gate_success": bool(gate_success), "gate_loss": float(gate_loss) if np.isfinite(gate_loss) else np.nan, "last_known_hold_rmse": hold_rmse, **metrics, **mean_weights})

        sample_idx = np.linspace(0, len(valid_frame) - 1, min(len(valid_frame), RESIDUAL_SAMPLE_PER_FOLD)).astype(int)
        sample = valid_frame.iloc[sample_idx].copy()
        residual_parts.append(pd.DataFrame({"fold": int(fold), "candidate": spec["candidate"], "wellname": sample["wellname"].to_numpy(), "row_idx": sample["row_idx"].to_numpy(), "eval_progress": sample["eval_progress"].to_numpy(float), "distance_log1p": sample["distance_log1p"].to_numpy(float), "selected_analog_distance": sample.get("selected_analog_distance", pd.Series(np.nan, index=sample.index)).to_numpy(float), "combined_stress_score": sample.get("combined_stress_score", pd.Series(np.nan, index=sample.index)).to_numpy(float), "y_true": y_valid_tvt[sample_idx], "baseline_tvt": baseline_tvt[sample_idx], "mean_pred_tvt": valid_last_known[sample_idx] + mean_delta[sample_idx], "y_pred": pred_tvt[sample_idx], "residual": y_valid_tvt[sample_idx] - pred_tvt[sample_idx], "correction": correction[sample_idx], "gate_shrink": gate_shrink}))

        if spec["mode"] == "gate":
            weight_sample = sample[["wellname", "row_idx", "eval_progress", "distance_log1p"]].copy()
            weight_sample["fold"] = int(fold)
            weight_sample["candidate"] = spec["candidate"]
            for idx, name in enumerate(component_names):
                weight_sample[f"weight_{name}"] = weights[sample_idx, idx]
            weight_parts.append(weight_sample)

fold_metrics = pd.DataFrame(fold_rows)
residual_sample = pd.concat(residual_parts, ignore_index=True)
gate_diagnostics = pd.DataFrame(gate_diag_rows)
gate_weight_sample = pd.concat(weight_parts, ignore_index=True) if weight_parts else pd.DataFrame()
fit_seconds = time.time() - start_time
summary = (
    fold_metrics.groupby(["candidate", "mode", "mean_model"], as_index=False)
    .agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), mae_mean=("mae", "mean"), r2_mean=("r2", "mean"), correction_sd_mean=("correction_sd", "mean"), gate_shrink_mean=("gate_shrink", "mean"), n_valid_rows=("n_valid_rows", "sum"), last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"), gate_success_rate=("gate_success", "mean"), weight_zero_mean=("weight_zero", "mean"), weight_hsgp_mean=("weight_hsgp", "mean"), weight_state_mean=("weight_state", "mean"), weight_blend_mean=("weight_blend", "mean"))
    .sort_values("rmse_mean")
    .reset_index(drop=True)
)
summary["rmse_delta_vs_hold"] = summary["rmse_mean"] - summary["last_known_hold_rmse_mean"]
huber_reference = float(summary.loc[summary["candidate"].eq("huber_mean_only"), "rmse_mean"].iloc[0])
summary["rmse_delta_vs_huber"] = summary["rmse_mean"] - huber_reference
summary["rmse_delta_vs_public_best"] = summary["rmse_mean"] - CONFIG["current_public_best"]
print({"fit_seconds": round(fit_seconds, 2), "fold_rows": len(fold_metrics), "sample_rows": len(residual_sample)})
summary


{'fit_seconds': 1718.11, 'fold_rows': 21, 'sample_rows': 46200}


,candidate,mode,mean_model,rmse_mean,rmse_std,mae_mean,r2_mean,correction_sd_mean,gate_shrink_mean,n_valid_rows,last_known_hold_rmse_mean,gate_success_rate,weight_zero_mean,weight_hsgp_mean,weight_state_mean,weight_blend_mean,rmse_delta_vs_hold,rmse_delta_vs_huber,rmse_delta_vs_public_best
0,huber_plus_gate_conservative,gate,huber,14.107248,0.543173,9.975809,0.999475,2.687916,0.994019,92760,15.569941,1.0,0.177083,0.202335,0.584058,0.036524,-1.462693,-0.398059,-0.364752
1,huber_plus_gate_medium,gate,huber,14.144056,0.552409,9.990675,0.999472,2.698478,0.995212,92760,15.569941,1.0,0.201081,0.187657,0.535985,0.075277,-1.425884,-0.361250,-0.327944
2,huber_plus_gate_open,gate,huber,14.178254,0.482565,10.015682,0.999470,2.730069,0.995572,92760,15.569941,1.0,0.173584,0.185036,0.509135,0.132246,-1.391687,-0.327053,-0.293746
3,huber_plus_state_s025,fixed,huber,14.328156,0.539958,10.068718,0.999459,0.956554,0.250000,92760,15.569941,1.0,NaN,NaN,NaN,NaN,-1.241785,-0.177151,-0.143844
4,huber_plus_hsgp_s025,fixed,huber,14.483751,0.569155,10.162605,0.999447,0.351657,0.250000,92760,15.569941,1.0,NaN,NaN,NaN,NaN,-1.086190,-0.021556,0.011751
5,huber_mean_only,mean,huber,14.505307,0.562634,10.169463,0.999445,0.000000,0.000000,92760,15.569941,1.0,NaN,NaN,NaN,NaN,-1.064634,0.000000,0.033307
6,gam_mean_only,mean,gam,25.817673,17.383099,11.571100,0.997786,0.000000,0.000000,92760,15.569941,1.0,NaN,NaN,NaN,NaN,10.247732,11.312366,11.345673


## 5. Stress Diagnostics And Artifacts

In [5]:

"Summarize stress slices, export tables, and write the model report."

best = summary.iloc[0]
best_candidate = str(best["candidate"])
residual_sample["progress_zone"] = pd.cut(residual_sample["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
if residual_sample["selected_analog_distance"].notna().sum() > 10:
    residual_sample["analog_distance_bin"] = pd.qcut(residual_sample["selected_analog_distance"], q=5, labels=["closest", "near", "middle", "far", "farthest"], duplicates="drop")
else:
    residual_sample["analog_distance_bin"] = "missing"

zone_summary = residual_sample.groupby(["candidate", "progress_zone"], observed=True).apply(lambda group: pd.Series({"n_rows": len(group), "rmse": rmse(group["y_true"], group["y_pred"]), "baseline_rmse": rmse(group["y_true"], group["baseline_tvt"]), "mean_residual": float(group["residual"].mean()), "mean_correction": float(group["correction"].mean())}), include_groups=False).reset_index()
zone_summary["rmse_delta_vs_hold"] = zone_summary["rmse"] - zone_summary["baseline_rmse"]
analog_bin_summary = residual_sample.groupby(["candidate", "analog_distance_bin"], observed=True).apply(lambda group: pd.Series({"n_rows": len(group), "n_wells": group["wellname"].nunique(), "rmse": rmse(group["y_true"], group["y_pred"]), "baseline_rmse": rmse(group["y_true"], group["baseline_tvt"]), "mean_correction": float(group["correction"].mean())}), include_groups=False).reset_index()
analog_bin_summary["rmse_delta_vs_hold"] = analog_bin_summary["rmse"] - analog_bin_summary["baseline_rmse"]
well_diagnostics = residual_sample.loc[residual_sample["candidate"].eq(best_candidate)].groupby("wellname").agg(n_rows=("y_true", "size"), mean_residual=("residual", "mean"), mean_abs_residual=("residual", lambda series: float(np.mean(np.abs(series)))), rmse=("residual", lambda series: float(np.sqrt(np.mean(np.square(series))))), mean_correction=("correction", "mean"), mean_progress=("eval_progress", "mean")).reset_index().sort_values("rmse", ascending=False)

paths = {
    "summary": TABLE_DIR / f"model__gated_bayesian_regime_summary__{RUN_DATE}.csv",
    "fold_metrics": TABLE_DIR / f"model__gated_bayesian_regime_fold_metrics__{RUN_DATE}.csv",
    "gate_diagnostics": TABLE_DIR / f"model__gated_bayesian_regime_gate_diagnostics__{RUN_DATE}.csv",
    "gate_weight_sample": TABLE_DIR / f"model__gated_bayesian_regime_gate_weight_sample__{RUN_DATE}.csv",
    "zone_summary": TABLE_DIR / f"model__gated_bayesian_regime_zone_summary__{RUN_DATE}.csv",
    "analog_bin_summary": TABLE_DIR / f"model__gated_bayesian_regime_analog_bin_summary__{RUN_DATE}.csv",
    "well_diagnostics": TABLE_DIR / f"model__gated_bayesian_regime_well_diagnostics__{RUN_DATE}.csv",
    "residual_sample": TABLE_DIR / f"model__gated_bayesian_regime_residual_sample__{RUN_DATE}.csv",
}
exports = {"summary": summary, "fold_metrics": fold_metrics, "gate_diagnostics": gate_diagnostics, "gate_weight_sample": gate_weight_sample, "zone_summary": zone_summary, "analog_bin_summary": analog_bin_summary, "well_diagnostics": well_diagnostics, "residual_sample": residual_sample}
for key, frame in exports.items():
    frame.to_csv(paths[key], index=False)
    if key != "residual_sample":
        shutil.copy2(paths[key], RUN_DIR / paths[key].name)

best_gate = summary.loc[summary["mode"].eq("gate")].sort_values("rmse_mean").iloc[0]
huber = summary.loc[summary["candidate"].eq("huber_mean_only")].iloc[0]
gam = summary.loc[summary["candidate"].eq("gam_mean_only")].iloc[0]
fixed_hsgp = summary.loc[summary["candidate"].eq("huber_plus_hsgp_s025")].iloc[0]
recommendation = "curiosity_package"
if float(best["rmse_mean"]) < CONFIG["current_local_xgb_safe"] - 0.05:
    recommendation = "package_as_candidate"
elif float(best_gate["rmse_mean"]) >= float(huber["rmse_mean"]):
    recommendation = "curiosity_only_negative_screen"

findings = pd.DataFrame([
    {"finding": "best_candidate", "value": best_candidate},
    {"finding": "best_strict_rmse", "value": f"{float(best['rmse_mean']):.4f}"},
    {"finding": "best_gate_candidate", "value": str(best_gate["candidate"])},
    {"finding": "best_gate_strict_rmse", "value": f"{float(best_gate['rmse_mean']):.4f}"},
    {"finding": "huber_mean_rmse", "value": f"{float(huber['rmse_mean']):.4f}"},
    {"finding": "gam_mean_rmse", "value": f"{float(gam['rmse_mean']):.4f}"},
    {"finding": "fixed_hsgp_rmse", "value": f"{float(fixed_hsgp['rmse_mean']):.4f}"},
    {"finding": "best_minus_huber", "value": f"{float(best['rmse_mean'] - huber['rmse_mean']):.4f}"},
    {"finding": "best_gate_minus_huber", "value": f"{float(best_gate['rmse_mean'] - huber['rmse_mean']):.4f}"},
    {"finding": "recommendation", "value": recommendation},
])
findings_path = TABLE_DIR / f"model__gated_bayesian_regime_findings__{RUN_DATE}.csv"
findings.to_csv(findings_path, index=False)
shutil.copy2(findings_path, RUN_DIR / findings_path.name)

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison_row = pd.DataFrame([{"run_id": RUN_ID, "model_family": "bayesian_gated_regime", "model_name": best_candidate, "cv_rmse_mean": float(best["rmse_mean"]), "cv_rmse_std": float(best["rmse_std"]), "train_time_s": float(fit_seconds), "n_features": int(len(feature_columns) + len(gate_feature_cols)), "notes": "Student-t MAP soft gate over zero, Hilbert, state, and blended residual experts with shrinkage priors.", "date": RUN_DATE, "validation_design": CONFIG["fold_column"], "local_rmse": float(best["rmse_mean"]), "stress_rmse": "", "public_lb_rmse": "", "submission_ref": ""}])
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    keep = ~comparison.get("run_id", pd.Series(dtype=str)).astype(str).eq(RUN_ID)
    comparison = pd.concat([comparison.loc[keep], comparison_row], ignore_index=True)
else:
    comparison = comparison_row
comparison.to_csv(comparison_path, index=False)
shutil.copy2(comparison_path, RUN_DIR / comparison_path.name)

report_path = REPORT_DIR / f"model__gated_bayesian_regime__{RUN_DATE}.md"
report_lines = [
    "# Gated Bayesian Stratigraphic-Regime Model",
    "",
    f"Run id: `{RUN_ID}`",
    "",
    "## Result",
    "",
    f"- Best candidate: `{best_candidate}`",
    f"- Best strict-hybrid RMSE: **{float(best['rmse_mean']):.4f}**",
    f"- Best gate candidate: `{best_gate['candidate']}` at **{float(best_gate['rmse_mean']):.4f}**",
    f"- Huber mean-only RMSE: **{float(huber['rmse_mean']):.4f}**",
    f"- GAM mean-only RMSE: **{float(gam['rmse_mean']):.4f}**",
    f"- Fixed HSGP shrinkage RMSE: **{float(fixed_hsgp['rmse_mean']):.4f}**",
    f"- Best minus Huber mean-only: **{float(best['rmse_mean'] - huber['rmse_mean']):.4f}**",
    f"- Recommendation: `{recommendation}`",
    "",
    "## Interpretation",
    "",
    "The model treats known-zone state as a gate over residual experts rather than as raw predictive columns. The priors deliberately shrink the gate toward low correction magnitude, so positive movement here is stronger evidence than an unconstrained residual learner.",
    "",
    "## Artifacts",
]
for key, path in paths.items():
    report_lines.append(f"- {key}: {path}")
report_lines.append(f"- findings: {findings_path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

with RunBundle(RUN_TAG, CONFIG, seed=SEED) as rb:
    rb.add_input(str(TRAIN_DIR))
    rb.add_input(str(descriptor_path))
    rb.add_metric("rows", float(len(feature_frame)))
    rb.add_metric("best_rmse", float(best["rmse_mean"]))
    rb.add_metric("best_gate_rmse", float(best_gate["rmse_mean"]))
    rb.add_metric("huber_mean_rmse", float(huber["rmse_mean"]))
    rb.add_metric("best_minus_huber", float(best["rmse_mean"] - huber["rmse_mean"]))
    for key, path in paths.items():
        if key != "residual_sample":
            rb.add_output(str(path))
    rb.add_output(str(findings_path))
    rb.add_output(str(report_path))
    rb.set_notes("Gated Bayesian regime screen with Student-t MAP mixture weights and shrinkage priors.")

print(report_path)
findings


C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\model__gated_bayesian_regime__2026-06-30.md


,finding,value
0,best_candidate,huber_plus_gate_conservative
1,best_strict_rmse,14.1072
2,best_gate_candidate,huber_plus_gate_conservative
3,best_gate_strict_rmse,14.1072
4,huber_mean_rmse,14.5053
5,gam_mean_rmse,25.8177
6,fixed_hsgp_rmse,14.4838
7,best_minus_huber,-0.3981
8,best_gate_minus_huber,-0.3981
9,recommendation,curiosity_package


## 6. Headline Tables

In [6]:

"Show the headline model ranking and gate diagnostics for review."

summary.head(12), gate_diagnostics.sort_values(["candidate", "fold"]).head(20), findings


(                      candidate   mode mean_model  rmse_mean   rmse_std   mae_mean   r2_mean  correction_sd_mean  gate_shrink_mean  n_valid_rows  last_known_hold_rmse_mean  gate_success_rate  weight_zero_mean  \
 0  huber_plus_gate_conservative   gate      huber  14.107248   0.543173   9.975809  0.999475            2.687916          0.994019         92760                  15.569941                1.0          0.177083   
 1        huber_plus_gate_medium   gate      huber  14.144056   0.552409   9.990675  0.999472            2.698478          0.995212         92760                  15.569941                1.0          0.201081   
 2          huber_plus_gate_open   gate      huber  14.178254   0.482565  10.015682  0.999470            2.730069          0.995572         92760                  15.569941                1.0          0.173584   
 3         huber_plus_state_s025  fixed      huber  14.328156   0.539958  10.068718  0.999459            0.956554          0.250000         92760   